In [ ]:
import numpy as np
import pandas as pd
import json
import time
import warnings
warnings.filterwarnings("ignore")

# ============================
# ML Libraries
# ============================
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, make_scorer

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from pytorch_tabnet.tab_model import TabNetRegressor

# ============================
# SETTINGS
# ============================
RANDOM_STATE = 42
N_SPLITS = 5
N_ITER = 30   # safe and acceptable for Q1
TARGET_COL = "yield"

TRAIN_PATH = "E:/Abroad period research/New idea for 2026/Corn Yield Estimation/yield_2006-24.csv"

# ============================
# LOAD DATA
# ============================
print("Loading training data...")
df = pd.read_csv(TRAIN_PATH)

assert TARGET_COL in df.columns, "Target column missing!"

# Drop non-feature columns if present
drop_cols = ['year', 'STATE', 'GEOID']
for col in drop_cols:
    if col in df.columns:
        df = df.drop(columns=[col])

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].values

# Scale once (important for TabNet fairness)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Data shape: {X.shape}")

# ============================
# COMMON UTILITIES
# ============================
rmse_scorer = make_scorer(
    lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)),
    greater_is_better=False
)

cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

best_params = {}

# =====================================================
# 1. XGBOOST OPTIMIZATION
# =====================================================
print("\n" + "="*60)
print("Optimizing XGBoost...")

xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_param_grid = {
    "n_estimators": [200, 300, 400, 500],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0.0, 0.1, 0.5, 1.0],
    "reg_lambda": [1.0, 1.5, 2.0, 5.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.2, 0.5]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_param_grid,
    n_iter=N_ITER,
    scoring=rmse_scorer,
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start = time.time()
xgb_search.fit(X_scaled, y)
print(f"XGBoost tuning time: {(time.time()-start)/60:.2f} min")

best_params["XGBoost"] = xgb_search.best_params_
print("Best XGBoost params:", best_params["XGBoost"])

# =====================================================
# 2. LIGHTGBM OPTIMIZATION
# =====================================================
print("\n" + "="*60)
print("Optimizing LightGBM...")

lgb_model = lgb.LGBMRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

lgb_param_grid = {
    "n_estimators": [200, 300, 400, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "num_leaves": [31, 63, 127, 255],
    "max_depth": [4, 6, 8, 10, -1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0.0, 0.1, 0.5, 1.0],
    "reg_lambda": [1.0, 1.5, 2.0, 5.0],
    "min_child_samples": [5, 10, 20, 30],
    "min_child_weight": [1e-3, 1e-2, 1e-1]
}

lgb_search = RandomizedSearchCV(
    estimator=lgb_model,
    param_distributions=lgb_param_grid,
    n_iter=N_ITER,
    scoring=rmse_scorer,
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start = time.time()
lgb_search.fit(X_scaled, y)
print(f"LightGBM tuning time: {(time.time()-start)/60:.2f} min")

best_params["LightGBM"] = lgb_search.best_params_
print("Best LightGBM params:", best_params["LightGBM"])

# =====================================================
# 3. CATBOOST OPTIMIZATION
# =====================================================
print("\n" + "="*60)
print("Optimizing CatBoost...")

cb_model = cb.CatBoostRegressor(
    random_state=RANDOM_STATE,
    verbose=0,
    thread_count=-1
)

cb_param_grid = {
    "iterations": [200, 300, 400, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "depth": [4, 6, 8, 10],
    "l2_leaf_reg": [1, 3, 5, 10],
    "border_count": [32, 64, 128, 255],
    "random_strength": [0.0, 0.1, 0.5, 1.0],
    "bagging_temperature": [0.0, 0.5, 1.0],
    "grow_policy": ["SymmetricTree", "Depthwise", "Lossguide"]
}

cb_search = RandomizedSearchCV(
    estimator=cb_model,
    param_distributions=cb_param_grid,
    n_iter=N_ITER,
    scoring=rmse_scorer,
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start = time.time()
cb_search.fit(X_scaled, y)
print(f"CatBoost tuning time: {(time.time()-start)/60:.2f} min")

best_params["CatBoost"] = cb_search.best_params_
print("Best CatBoost params:", best_params["CatBoost"])

# =====================================================
# 4. TABNET OPTIMIZATION (CONTROLLED GRID)
# =====================================================
print("\n" + "="*60)
print("Optimizing TabNet...")

tabnet_results = []
tabnet_grid = [
    {"n_d": 8, "n_a": 8, "n_steps": 3, "gamma": 1.3, "lambda_sparse": 1e-3, "lr": 0.02},
    {"n_d": 16, "n_a": 16, "n_steps": 3, "gamma": 1.3, "lambda_sparse": 1e-3, "lr": 0.02},
    {"n_d": 32, "n_a": 32, "n_steps": 3, "gamma": 1.3, "lambda_sparse": 1e-3, "lr": 0.02},
    {"n_d": 64, "n_a": 64, "n_steps": 4, "gamma": 1.5, "lambda_sparse": 1e-4, "lr": 0.01},
    {"n_d": 32, "n_a": 32, "n_steps": 5, "gamma": 1.2, "lambda_sparse": 1e-3, "lr": 0.005},
]

for cfg_idx, cfg in enumerate(tabnet_grid):
    print(f"\n  TabNet config {cfg_idx+1}/{len(tabnet_grid)}: {cfg}")
    rmses = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_scaled)):
        X_tr, X_val = X_scaled[train_idx], X_scaled[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model = TabNetRegressor(
            n_d=cfg["n_d"],
            n_a=cfg["n_a"],
            n_steps=cfg["n_steps"],
            gamma=cfg["gamma"],
            lambda_sparse=cfg["lambda_sparse"],
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=cfg["lr"]),
            mask_type='entmax',
            scheduler_params={
                "mode": "min",
                "patience": 10,
                "min_lr": 1e-5,
                "factor": 0.5,
            },
            scheduler_fn=optim.lr_scheduler.ReduceLROnPlateau,
            verbose=0,
            seed=RANDOM_STATE
        )

        try:
            model.fit(
                X_tr, y_tr.reshape(-1, 1),
                eval_set=[(X_val, y_val.reshape(-1, 1))],
                max_epochs=50,
                patience=10,
                batch_size=1024,
                virtual_batch_size=128,
                eval_metric=["rmse"]
            )

            preds = model.predict(X_val).flatten()
            rmse = np.sqrt(mean_squared_error(y_val, preds))
            rmses.append(rmse)
        except Exception as e:
            print(f"    Fold {fold_idx+1} failed: {e}")
            rmses.append(np.inf)
    
    if len(rmses) > 0 and not all(np.isinf(rmses)):
        avg_rmse = np.mean([r for r in rmses if not np.isinf(r)])
        tabnet_results.append((cfg, avg_rmse))
        print(f"  Config RMSE: {avg_rmse:.4f}")
    else:
        print(f"  Config failed for all folds")

if tabnet_results:
    best_tabnet = sorted(tabnet_results, key=lambda x: x[1])[0]
    best_params["TabNet"] = best_tabnet[0]
    print("\nBest TabNet params:", best_params["TabNet"])
else:
    best_params["TabNet"] = tabnet_grid[0]
    print("\nUsing default TabNet params:", best_params["TabNet"])

# =====================================================
# 5. NODE (NEURAL NETWORK) OPTIMIZATION
# =====================================================
print("\n" + "="*60)
print("Optimizing NODE (Neural Network)...")

# Convert to PyTorch tensors
X_tensor = torch.FloatTensor(X_scaled)
y_tensor = torch.FloatTensor(y).reshape(-1, 1)

# Define NODE model class (simplified version)
class SimpleNODE(nn.Module):
    def __init__(self, input_dim, hidden_dims=[256, 128], dropout_rate=0.2):
        super(SimpleNODE, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        self.model = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.model(x)

node_results = []
node_grid = [
    {"hidden_dims": [256, 128], "dropout_rate": 0.2, "lr": 0.001, "batch_size": 512},
    {"hidden_dims": [512, 256, 128], "dropout_rate": 0.3, "lr": 0.001, "batch_size": 256},
    {"hidden_dims": [256, 128, 64], "dropout_rate": 0.1, "lr": 0.0005, "batch_size": 1024},
    {"hidden_dims": [512, 256], "dropout_rate": 0.2, "lr": 0.0005, "batch_size": 512},
]

for cfg_idx, cfg in enumerate(node_grid):
    print(f"\n  NODE config {cfg_idx+1}/{len(node_grid)}")
    rmses = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_scaled)):
        try:
            X_tr, X_val = X_scaled[train_idx], X_scaled[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]
            
            # Convert to tensors
            X_tr_tensor = torch.FloatTensor(X_tr)
            y_tr_tensor = torch.FloatTensor(y_tr).reshape(-1, 1)
            X_val_tensor = torch.FloatTensor(X_val)
            y_val_tensor = torch.FloatTensor(y_val).reshape(-1, 1)
            
            # Create model
            model = SimpleNODE(
                input_dim=X_scaled.shape[1],
                hidden_dims=cfg["hidden_dims"],
                dropout_rate=cfg["dropout_rate"]
            )
            
            # Loss and optimizer
            criterion = nn.MSELoss()
            optimizer = optim.Adam(model.parameters(), lr=cfg["lr"])
            
            # Data loader
            train_dataset = TensorDataset(X_tr_tensor, y_tr_tensor)
            train_loader = DataLoader(train_dataset, batch_size=cfg["batch_size"], shuffle=True)
            
            # Training loop (simplified for HPO)
            model.train()
            for epoch in range(50):
                epoch_loss = 0
                for batch_X, batch_y in train_loader:
                    optimizer.zero_grad()
                    predictions = model(batch_X)
                    loss = criterion(predictions, batch_y)
                    loss.backward()
                    optimizer.step()
                    epoch_loss += loss.item()
            
            # Evaluation
            model.eval()
            with torch.no_grad():
                preds = model(X_val_tensor).numpy().flatten()
            
            rmse = np.sqrt(mean_squared_error(y_val, preds))
            rmses.append(rmse)
            
        except Exception as e:
            print(f"    Fold {fold_idx+1} failed: {e}")
            rmses.append(np.inf)
    
    if len(rmses) > 0 and not all(np.isinf(rmses)):
        avg_rmse = np.mean([r for r in rmses if not np.isinf(r)])
        node_results.append((cfg, avg_rmse))
        print(f"  Config RMSE: {avg_rmse:.4f}")

if node_results:
    best_node = sorted(node_results, key=lambda x: x[1])[0]
    best_params["NODE"] = best_node[0]
    print("\nBest NODE params:", best_params["NODE"])
else:
    # Default NODE parameters
    best_params["NODE"] = {
        "hidden_dims": [512, 256, 128],
        "dropout_rate": 0.2,
        "lr": 0.001,
        "batch_size": 512
    }
    print("\nUsing default NODE params:", best_params["NODE"])

# =====================================================
# 6. FT-TRANSFORMER OPTIMIZATION
# =====================================================
print("\n" + "="*60)
print("Optimizing FT-Transformer...")

# Define simple FT-Transformer class for HPO
class SimpleFTTransformer(nn.Module):
    def __init__(self, input_dim, d_model=256, nhead=8, num_layers=4, dropout=0.1):
        super(SimpleFTTransformer, self).__init__()
        
        self.embedding = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_head = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model//2, 1)
        )
        
    def forward(self, x):
        x = self.embedding(x)
        x = x.unsqueeze(1)
        x = self.transformer(x)
        x = x.mean(dim=1)
        x = self.output_head(x)
        return x

ft_transformer_results = []
ft_transformer_grid = [
    {"d_model": 128, "nhead": 4, "num_layers": 2, "dropout": 0.1, "lr": 0.0005, "batch_size": 128},
    {"d_model": 256, "nhead": 8, "num_layers": 4, "dropout": 0.1, "lr": 0.0005, "batch_size": 64},
    {"d_model": 128, "nhead": 8, "num_layers": 3, "dropout": 0.2, "lr": 0.001, "batch_size": 128},
    {"d_model": 256, "nhead": 4, "num_layers": 3, "dropout": 0.1, "lr": 0.0005, "batch_size": 64},
]

for cfg_idx, cfg in enumerate(ft_transformer_grid):
    print(f"\n  FT-Transformer config {cfg_idx+1}/{len(ft_transformer_grid)}")
    rmses = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_scaled)):
        try:
            X_tr, X_val = X_scaled[train_idx], X_scaled[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]
            
            # Convert to tensors
            X_tr_tensor = torch.FloatTensor(X_tr)
            y_tr_tensor = torch.FloatTensor(y_tr).reshape(-1, 1)
            X_val_tensor = torch.FloatTensor(X_val)
            y_val_tensor = torch.FloatTensor(y_val).reshape(-1, 1)
            
            # Create model
            model = SimpleFTTransformer(
                input_dim=X_scaled.shape[1],
                d_model=cfg["d_model"],
                nhead=cfg["nhead"],
                num_layers=cfg["num_layers"],
                dropout=cfg["dropout"]
            )
            
            # Loss and optimizer
            criterion = nn.MSELoss()
            optimizer = optim.Adam(model.parameters(), lr=cfg["lr"])
            
            # Data loader
            train_dataset = TensorDataset(X_tr_tensor, y_tr_tensor)
            train_loader = DataLoader(train_dataset, batch_size=cfg["batch_size"], shuffle=True)
            
            # Training loop
            model.train()
            for epoch in range(30):  # Shorter training for HPO
                epoch_loss = 0
                for batch_X, batch_y in train_loader:
                    optimizer.zero_grad()
                    predictions = model(batch_X)
                    loss = criterion(predictions, batch_y)
                    loss.backward()
                    optimizer.step()
                    epoch_loss += loss.item()
            
            # Evaluation
            model.eval()
            with torch.no_grad():
                preds = model(X_val_tensor).numpy().flatten()
            
            rmse = np.sqrt(mean_squared_error(y_val, preds))
            rmses.append(rmse)
            
        except Exception as e:
            print(f"    Fold {fold_idx+1} failed: {e}")
            rmses.append(np.inf)
    
    if len(rmses) > 0 and not all(np.isinf(rmses)):
        avg_rmse = np.mean([r for r in rmses if not np.isinf(r)])
        ft_transformer_results.append((cfg, avg_rmse))
        print(f"  Config RMSE: {avg_rmse:.4f}")

if ft_transformer_results:
    best_ft_transformer = sorted(ft_transformer_results, key=lambda x: x[1])[0]
    best_params["FT-Transformer"] = best_ft_transformer[0]
    print("\nBest FT-Transformer params:", best_params["FT-Transformer"])
else:
    # Default FT-Transformer parameters
    best_params["FT-Transformer"] = {
        "d_model": 256,
        "nhead": 8,
        "num_layers": 4,
        "dropout": 0.1,
        "lr": 0.0005,
        "batch_size": 64
    }
    print("\nUsing default FT-Transformer params:", best_params["FT-Transformer"])

# =====================================================
# 7. ENSEMBLE DEFAULT PARAMETERS
# =====================================================
print("\n" + "="*60)
print("Setting up Ensemble default parameters...")

# For ensemble models, we'll use simple defaults
best_params["Mean_Ensemble"] = {"method": "mean"}
best_params["Median_Ensemble"] = {"method": "median"}
best_params["Weighted_Ensemble"] = {"method": "weighted"}

# =====================================================
# SAVE RESULTS
# =====================================================
print("\n" + "="*60)
print("Saving optimized hyperparameters...")

# Save the best parameters
with open("best_hyperparameters.json", "w") as f:
    json.dump(best_params, f, indent=4)

# Also save as a more readable version
readable_params = {}
for model, params in best_params.items():
    readable_params[model] = params

with open("best_hyperparameters_readable.txt", "w") as f:
    f.write("="*80 + "\n")
    f.write("OPTIMIZED HYPERPARAMETERS FOR ALL MODELS\n")
    f.write("="*80 + "\n\n")
    
    for model, params in readable_params.items():
        f.write(f"{'='*60}\n")
        f.write(f"MODEL: {model}\n")
        f.write(f"{'='*60}\n")
        
        if isinstance(params, dict):
            for key, value in params.items():
                f.write(f"  {key}: {value}\n")
        else:
            f.write(f"  {params}\n")
        f.write("\n")

# Save a Python script to load these parameters
load_script = """
import json
import numpy as np

def load_optimized_hyperparameters():
    \"\"\"
    Load optimized hyperparameters from HPO run
    \"\"\"
    with open("best_hyperparameters.json", "r") as f:
        best_params = json.load(f)
    
    # Convert string representations back to proper types
    for model in best_params:
        if isinstance(best_params[model], dict):
            for key in best_params[model]:
                val = best_params[model][key]
                if isinstance(val, str):
                    # Try to convert string representations
                    if val == "True":
                        best_params[model][key] = True
                    elif val == "False":
                        best_params[model][key] = False
                    elif val == "None":
                        best_params[model][key] = None
                    elif "." in val and val.replace(".", "").replace("-", "").isdigit():
                        try:
                            best_params[model][key] = float(val)
                        except:
                            pass
                    elif val.isdigit() or (val.startswith("-") and val[1:].isdigit()):
                        try:
                            best_params[model][key] = int(val)
                        except:
                            pass
    
    return best_params

# Usage:
# best_params = load_optimized_hyperparameters()
# xgb_params = best_params.get("XGBoost", {})
# lgb_params = best_params.get("LightGBM", {})
# etc...
"""

with open("load_optimized_params.py", "w") as f:
    f.write(load_script)

print("\n✅ Hyperparameter optimization completed for ALL models!")
print("📁 Saved to:")
print("   - best_hyperparameters.json (JSON format)")
print("   - best_hyperparameters_readable.txt (Human readable)")
print("   - load_optimized_params.py (Helper script)")

print("\n" + "="*80)
print("SUMMARY OF OPTIMIZED PARAMETERS")
print("="*80)

for model, params in best_params.items():
    print(f"\n{model}:")
    if isinstance(params, dict):
        for key, value in list(params.items())[:5]:  # Show first 5 params
            print(f"  {key}: {value}")
        if len(params) > 5:
            print(f"  ... and {len(params)-5} more parameters")
    else:
        print(f"  {params}")



# Total time
total_time = time.time() - start
print(f"\n  Total HPO time: {total_time/60:.2f} minutes")